In [194]:
import torch
import torch.nn as nn
import torchcvnn

In [195]:
import torch

data = torch.load("data/train_blip_features.pt")

features = data["features"]
questions = data["questions"]
labels = data["labels"]


In [196]:
from torch.utils.data import Dataset

class DisasterVQADataset(Dataset):

    def __init__(self, pt_file):

        data = torch.load(pt_file)

        self.features = data["features"]
        self.questions = data["questions"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.features[idx],
            self.questions[idx],
            self.labels[idx]
        )

In [197]:
from torch.utils.data import DataLoader

train_dataset = DisasterVQADataset(
    "data/train_blip_features.pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

features, questions, labels = next(
    iter(train_loader)
)

In [198]:
import torch.nn as nn

class SemanticEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(768,512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )


    def forward(self, x):

        return self.encoder(x)

In [199]:
def real_to_complex(x):

    real = x[..., ::2]
    imag = x[..., 1::2]

    return torch.complex(real, imag)

def complex_awgn(complex_signal, snr_db):

    signal_power = torch.mean(
        torch.abs(complex_signal) ** 2
    )

    snr_linear = 10 ** (snr_db / 10)

    noise_power = signal_power / snr_linear

    noise_std = torch.sqrt(noise_power / 2)

    noise_real = (
        torch.randn_like(complex_signal.real)
        * noise_std
    )

    noise_imag = (
        torch.randn_like(complex_signal.imag)
        * noise_std
    )

    noise = torch.complex(
        noise_real,
        noise_imag
    )

    return complex_signal + noise

def complex_to_real(z):

    real = z.real
    imag = z.imag

    return torch.cat(
        [real, imag],
        dim=-1
    )

In [200]:
class SemanticDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 768)
        )

    def forward(self, x):

        return self.decoder(x)

In [201]:
class QuestionEncoder(nn.Module):

    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, questions):

        embedded = self.embedding(
            questions
        )

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        question_vector = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        return question_vector

In [202]:
import pickle

with open("data/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

val_dataset = DisasterVQADataset(
    "data/val_blip_features.pt"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [203]:
import torch
import torch.nn as nn
import torchcvnn.nn as c_nn

layer = nn.Linear(128, 64).to(torch.cfloat)

x = torch.randn(4, 128, dtype=torch.cfloat)

y = layer(x)

print(y.dtype)
print(y.shape)

torch.complex64
torch.Size([4, 64])


C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\3223998043.py:5: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  layer = nn.Linear(128, 64).to(torch.cfloat)


In [204]:
class ComplexSemanticEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            384,
            256
        ).to(torch.cfloat)

        self.act1 = c_nn.Cardioid()

        self.fc2 = nn.Linear(
            256,
            128
        ).to(torch.cfloat)

        self.act2 = c_nn.Cardioid()

        self.fc3 = nn.Linear(
            128,
            64
        ).to(torch.cfloat)

    def forward(self, x):

        x = real_to_complex(x)

        x = self.fc1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.act2(x)

        x = self.fc3(x)

        return x

In [205]:
class ComplexSemanticDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            64,
            128
        ).to(torch.cfloat)

        self.act1 = c_nn.Cardioid()

        self.fc2 = nn.Linear(
            128,
            256
        ).to(torch.cfloat)

        self.act2 = c_nn.Cardioid()

        self.fc3 = nn.Linear(
            256,
            384
        ).to(torch.cfloat)

    def forward(self, x):

        x = self.fc1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.act2(x)

        x = self.fc3(x)

        x = complex_to_real(x)

        return x

In [206]:
class VQAClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        return self.classifier(x)

In [207]:
import torchcvnn.nn as c_nn

print(c_nn.CReLU)

<class 'torchcvnn.nn.modules.activation.CReLU'>


In [208]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = ComplexSemanticEncoder().to(device)
decoder = ComplexSemanticDecoder().to(device)

features, questions, labels = next(iter(train_loader))

features = features.to(device)

compressed = encoder(features)
print(compressed.shape)

received = complex_awgn(
    compressed,
    snr_db=10
)

reconstructed = decoder(received)
print(reconstructed.shape)

torch.Size([32, 32, 64])
torch.Size([32, 32, 768])


C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:9: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:16: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:23: UserWarning: Complex modules are a new feature under active development whose d

In [209]:
encoder = ComplexSemanticEncoder().to(device)
decoder = ComplexSemanticDecoder().to(device)

features, questions, labels = next(iter(train_loader))

features = features.to(device)

compressed = encoder(features)
print("compressed:", compressed.shape, compressed.dtype)

received = complex_awgn(
    compressed,
    snr_db=10
)

print("received:", received.shape, received.dtype)

reconstructed = decoder(received)
print("reconstructed:", reconstructed.shape, reconstructed.dtype)

compressed: torch.Size([32, 32, 64]) torch.complex64
received: torch.Size([32, 32, 64]) torch.complex64
reconstructed: torch.Size([32, 32, 768]) torch.float32


C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:9: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:16: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:23: UserWarning: Complex modules are a new feature under active development whose d

In [210]:
class CVNNVQA(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.encoder = ComplexSemanticEncoder()

        self.decoder = ComplexSemanticDecoder()

        self.question_encoder = QuestionEncoder(
            vocab_size=vocab_size
        )

        self.classifier = VQAClassifier()

    def forward(self, image_features, questions):

        compressed = self.encoder(
            image_features
        )

        received = complex_awgn(
            compressed,
            snr_db=10
        )

        reconstructed = self.decoder(
            received
        )

        image_vector = reconstructed.mean(
            dim=1
        )

        question_vector = self.question_encoder(
            questions
        )

        fused = torch.cat(
            [image_vector, question_vector],
            dim=1
        )

        logits = self.classifier(
            fused
        )

        return logits

In [211]:
model = CVNNVQA(
    vocab_size=len(word2idx)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:9: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:16: UserWarning: Complex modules are a new feature under active development whose design may change, and some modules might not work as expected when using complex tensors as parameters or buffers. Please file an issue at https://github.com/pytorch/pytorch/issues/new?template=bug-report.yml if a complex module does not work as expected.
  ).to(torch.cfloat)
C:\Users\Shourya\AppData\Local\Temp\ipykernel_28336\2004815120.py:23: UserWarning: Complex modules are a new feature under active development whose d

In [212]:
num_epochs = 30

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for features, questions, labels in train_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            features,
            questions
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    train_acc = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Acc: {train_acc:.2f}%"
    )

Epoch 1/30 | Train Acc: 70.21%
Epoch 2/30 | Train Acc: 70.21%
Epoch 3/30 | Train Acc: 72.76%
Epoch 4/30 | Train Acc: 79.44%
Epoch 5/30 | Train Acc: 81.30%
Epoch 6/30 | Train Acc: 82.35%
Epoch 7/30 | Train Acc: 83.68%
Epoch 8/30 | Train Acc: 85.31%
Epoch 9/30 | Train Acc: 86.12%
Epoch 10/30 | Train Acc: 86.93%
Epoch 11/30 | Train Acc: 89.20%
Epoch 12/30 | Train Acc: 90.19%
Epoch 13/30 | Train Acc: 91.64%
Epoch 14/30 | Train Acc: 91.52%
Epoch 15/30 | Train Acc: 92.74%
Epoch 16/30 | Train Acc: 94.02%
Epoch 17/30 | Train Acc: 94.31%
Epoch 18/30 | Train Acc: 95.30%
Epoch 19/30 | Train Acc: 95.93%
Epoch 20/30 | Train Acc: 96.17%
Epoch 21/30 | Train Acc: 96.17%
Epoch 22/30 | Train Acc: 96.69%
Epoch 23/30 | Train Acc: 97.04%
Epoch 24/30 | Train Acc: 97.56%
Epoch 25/30 | Train Acc: 97.97%
Epoch 26/30 | Train Acc: 97.68%
Epoch 27/30 | Train Acc: 96.92%
Epoch 28/30 | Train Acc: 96.86%
Epoch 29/30 | Train Acc: 98.32%
Epoch 30/30 | Train Acc: 98.61%


In [213]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for features, questions, labels in val_loader:

        features = features.to(device)
        questions = questions.to(device)
        labels = labels.to(device)

        logits = model(
            features,
            questions
        )

        preds = logits.argmax(dim=1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

val_accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Validation Accuracy: 74.25%
